# ModernBERT V3b Native Full-Text GECS Training

**Note:** V3a over-balanced the long tail and underperformed V2 by epoch 5. V3b removes weighted sampling, restores 512-token context, and returns to the V2-stable loss balance while keeping full-text input and dev-only hierarchy tuning.

Goal: push Task 1 macro F1 toward 80%+ without test-label restoration.

This run uses the findings from V2/V3 safely:
- Full `text` column, not segment-only truncation.
- ModernBERT-base backbone.
- Multi-task heads for sector, group, and industry.
- Class-balanced sampling plus effective-number class weights.
- Hierarchy-aware inference tuned only on the dev split.
- Locked test set evaluated once after the dev-tuned recipe is fixed.

Upload these files when prompted:
- `task1_train.csv`
- `task1_test.csv`
- optional `gecs_taxonomy.json`

In [1]:
# 1. Environment setup
!pip -q install -U "transformers>=4.48.0" datasets accelerate scikit-learn pandas numpy tqdm pytorch-optimizer

import json
import math
import random
import time
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch

SEED = 42
REQUIRE_A100 = True
REQUIRE_BF16 = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())
    if REQUIRE_A100 and "A100" not in torch.cuda.get_device_name(0):
        raise RuntimeError("Reconnect Colab to an A100 runtime before running V3.")
    if REQUIRE_BF16 and not torch.cuda.is_bf16_supported():
        raise RuntimeError("This V3 recipe is locked to bf16. Reconnect to an A100 runtime.")



ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
tsfresh 0.21.1 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.12.0 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.3 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires scipy>=1.13, but you have scipy 1.12.0 which is incompatible.
access 1.1.10.post3 requires scipy>=1.14.1, but you have scipy 1.12.0 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is i

In [2]:
# 2. Upload data
from google.colab import files

RUN_NAME = "modernbert_gecs_v3b_fulltext_stable"
OUT_DIR = Path("/content") / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = Path("/content/task1_train.csv")
TEST_PATH = Path("/content/task1_test.csv")

if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    print("Upload task1_train.csv, task1_test.csv, and optionally gecs_taxonomy.json")
    uploaded = files.upload()
    print("Uploaded:", list(uploaded.keys()))
else:
    print("Files already exist. Skipping upload.")

if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError("Expected task1_train.csv and task1_test.csv in /content.")

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
print("train:", train_df.shape, "test:", test_df.shape)
print("train columns:", list(train_df.columns))

Files already exist. Skipping upload.
train: (42868, 3) test: (10717, 3)
train columns: ['text', 'label_idx', 'mstar_code']


In [11]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='huggingface_hub')

# IMPORTANT: Downgrading numpy/scipy must be done in a separate cell,
# followed by a RUNTIME RESTART, before importing sklearn.
# !pip install numpy==1.26.4 pandas==2.2.2 scipy==1.12.0 -q --force-reinstall

# 3. Full-text labels and leakage guard
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import json
import pandas as pd
import numpy as np

def norm_code(value):
    return str(int(value)).zfill(8)

def clean_text(value):
    text = str(value or "").replace("\n", " ")
    return " ".join(text.split())

required = {"text", "mstar_code"}
missing_train = required - set(train_df.columns)
missing_test = required - set(test_df.columns)
if missing_train or missing_test:
    raise RuntimeError(f"Missing columns: train={missing_train}, test={missing_test}")

for frame in (train_df, test_df):
    frame["text"] = frame["text"].map(clean_text)
    frame["industry_code"] = frame["mstar_code"].map(norm_code)
    frame["sector_code"] = frame["industry_code"].str[:3]
    frame["group_code"] = frame["industry_code"].str[:5]

train_df = train_df[train_df["text"].str.len() > 0].drop_duplicates(subset=["text", "industry_code"]).copy()
test_df = test_df[test_df["text"].str.len() > 0].copy()

overlap = set(train_df["text"]).intersection(set(test_df["text"]))
leakage_report = {
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "exact_text_overlap_count": int(len(overlap)),
    "exact_text_overlap_rate_vs_test": float(len(overlap) / max(1, len(test_df))),
    "note": "V3 does not use test labels during training or dev tuning. Exact overlap should be investigated if non-zero.",
}
print(json.dumps(leakage_report, indent=2))

le_sector = LabelEncoder().fit(pd.concat([train_df["sector_code"], test_df["sector_code"]], axis=0))
le_group = LabelEncoder().fit(pd.concat([train_df["group_code"], test_df["group_code"]], axis=0))
le_industry = LabelEncoder().fit(pd.concat([train_df["industry_code"], test_df["industry_code"]], axis=0))

for frame in (train_df, test_df):
    frame["sector_idx"] = le_sector.transform(frame["sector_code"])
    frame["group_idx"] = le_group.transform(frame["group_code"])
    frame["industry_idx"] = le_industry.transform(frame["industry_code"])

N_SECTORS = len(le_sector.classes_)
N_GROUPS = len(le_group.classes_)
N_INDUSTRIES = len(le_industry.classes_)

try:
    train_fit_df, dev_df = train_test_split(
        train_df,
        test_size=0.10,
        random_state=SEED,
        stratify=train_df["industry_code"],
    )
except ValueError:
    train_fit_df, dev_df = train_test_split(train_df, test_size=0.10, random_state=SEED, shuffle=True)

train_fit_df = train_fit_df.reset_index(drop=True)
dev_df = dev_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("classes:", {"sector": N_SECTORS, "group": N_GROUPS, "industry": N_INDUSTRIES})
print("splits:", {"train_fit": len(train_fit_df), "dev": len(dev_df), "official_test": len(test_df)})
print("word length train:")
print(train_fit_df["text"].str.split().str.len().describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

{
  "train_rows": 42319,
  "test_rows": 10717,
  "exact_text_overlap_count": 248,
  "exact_text_overlap_rate_vs_test": 0.02314080432956984,
  "note": "V3 does not use test labels during training or dev tuning. Exact overlap should be investigated if non-zero."
}
classes: {'sector': 11, 'group': 55, 'industry': 145}
splits: {'train_fit': 38087, 'dev': 4232, 'official_test': 10717}
word length train:
count    38087.000000
mean        88.850815
std         35.030055
min          7.000000
50%         88.000000
90%        135.000000
95%        149.000000
99%        173.000000
max        207.000000
Name: text, dtype: float64


In [12]:
# 4. Run config
SMOKE_TEST = False
SMOKE_SAMPLES = 800

MODEL_NAME = "answerdotai/ModernBERT-large"
MAX_LEN = 512
FULL_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 3
MIN_DEV_F1_DELTA = 0.001
BATCH_SIZE = 8
GRAD_ACCUM = 8
ENCODER_LR = 5e-6
HEAD_LR = 5e-4
WARMUP_RATIO = 0.05
LABEL_SMOOTHING = 0.02

if SMOKE_TEST:
    train_run_df = train_fit_df.sample(min(SMOKE_SAMPLES, len(train_fit_df)), random_state=SEED).copy()
    dev_run_df = dev_df.sample(min(SMOKE_SAMPLES, len(dev_df)), random_state=SEED).copy()
    test_run_df = test_df.copy()
    EPOCHS = 1
else:
    train_run_df = train_fit_df.copy()
    dev_run_df = dev_df.copy()
    test_run_df = test_df.copy()
    EPOCHS = FULL_EPOCHS

print(json.dumps({
    "run_name": RUN_NAME,
    "model": MODEL_NAME,
    "max_len": MAX_LEN,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM,
    "encoder_lr": ENCODER_LR,
    "head_lr": HEAD_LR,
    "smoke_test": SMOKE_TEST,
}, indent=2))

{
  "run_name": "modernbert_gecs_v3b_fulltext_stable",
  "model": "answerdotai/ModernBERT-large",
  "max_len": 512,
  "epochs": 10,
  "batch_size": 8,
  "grad_accum": 8,
  "encoder_lr": 5e-06,
  "head_lr": 0.0005,
  "smoke_test": false
}


In [13]:
# 5. Tokenization
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

def to_dataset(df):
    return Dataset.from_pandas(
        df[["text", "sector_idx", "group_idx", "industry_idx"]],
        preserve_index=False,
    )

train_ds = to_dataset(train_run_df)
dev_ds = to_dataset(dev_run_df)
test_ds = to_dataset(test_run_df)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LEN)

train_ds = train_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
dev_ds = dev_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
test_ds = test_ds.map(tokenize_batch, batched=True, remove_columns=["text"])

columns = ["input_ids", "attention_mask", "sector_idx", "group_idx", "industry_idx"]
train_ds.set_format(type="torch", columns=columns)
dev_ds.set_format(type="torch", columns=columns)
test_ds.set_format(type="torch", columns=columns)

print(train_ds)
print(dev_ds)
print(test_ds)



Map:   0%|          | 0/38087 [00:00<?, ? examples/s]

Map:   0%|          | 0/4232 [00:00<?, ? examples/s]

Map:   0%|          | 0/10717 [00:00<?, ? examples/s]

Dataset({
    features: ['sector_idx', 'group_idx', 'industry_idx', 'input_ids', 'attention_mask'],
    num_rows: 38087
})
Dataset({
    features: ['sector_idx', 'group_idx', 'industry_idx', 'input_ids', 'attention_mask'],
    num_rows: 4232
})
Dataset({
    features: ['sector_idx', 'group_idx', 'industry_idx', 'input_ids', 'attention_mask'],
    num_rows: 10717
})


In [6]:
# 6. Model, class weights, and hierarchy maps
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoModel, get_cosine_schedule_with_warmup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def effective_num_weights(labels, num_classes, beta=0.9997, power=0.6):
    counts = np.bincount(labels, minlength=num_classes).astype(np.float64)
    eff_num = 1.0 - np.power(beta, counts)
    weights = (1.0 - beta) / np.clip(eff_num, 1e-12, None)
    weights = np.power(weights, power)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)

sector_weights = effective_num_weights(train_run_df["sector_idx"].to_numpy(), N_SECTORS, beta=0.999, power=0.25)
group_weights = effective_num_weights(train_run_df["group_idx"].to_numpy(), N_GROUPS, beta=0.9995, power=0.35)
industry_weights = effective_num_weights(train_run_df["industry_idx"].to_numpy(), N_INDUSTRIES, beta=0.9997, power=0.45)

industry_to_group_idx = []
industry_to_sector_idx = []
for code in le_industry.classes_:
    industry_to_group_idx.append(int(le_group.transform([code[:5]])[0]))
    industry_to_sector_idx.append(int(le_sector.transform([code[:3]])[0]))
industry_to_group_idx = torch.tensor(industry_to_group_idx, dtype=torch.long, device=device)
industry_to_sector_idx = torch.tensor(industry_to_sector_idx, dtype=torch.long, device=device)

class MultiTaskModernBERT(nn.Module):
    def __init__(self, model_name, n_sectors, n_groups, n_industries, dropout=0.10):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name, trust_remote_code=True)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(hidden)
        self.sector_head = nn.Linear(hidden, n_sectors)
        self.group_head = nn.Linear(hidden, n_groups)
        self.industry_head = nn.Linear(hidden, n_industries)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output if getattr(outputs, "pooler_output", None) is not None else outputs.last_hidden_state[:, 0]
        pooled = self.dropout(self.norm(pooled))
        return {
            "sector_logits": self.sector_head(pooled),
            "group_logits": self.group_head(pooled),
            "industry_logits": self.industry_head(pooled),
        }

model = MultiTaskModernBERT(MODEL_NAME, N_SECTORS, N_GROUPS, N_INDUSTRIES).to(device)

loss_sector = nn.CrossEntropyLoss(weight=sector_weights.to(device), label_smoothing=LABEL_SMOOTHING)
loss_group = nn.CrossEntropyLoss(weight=group_weights.to(device), label_smoothing=LABEL_SMOOTHING)
loss_industry = nn.CrossEntropyLoss(weight=industry_weights.to(device), label_smoothing=LABEL_SMOOTHING)
alpha, beta_loss, gamma = 0.2, 0.3, 0.5

encoder_params, head_params = [], []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if name.startswith("encoder."):
        encoder_params.append(param)
    else:
        head_params.append(param)

try:
    from pytorch_optimizer import StableAdamW
    optimizer = StableAdamW([
        {"params": encoder_params, "lr": ENCODER_LR, "weight_decay": 0.01},
        {"params": head_params, "lr": HEAD_LR, "weight_decay": 0.01},
    ])
    optimizer_name = "StableAdamW"
except Exception:
    optimizer = torch.optim.AdamW([
        {"params": encoder_params, "lr": ENCODER_LR, "weight_decay": 0.01},
        {"params": head_params, "lr": HEAD_LR, "weight_decay": 0.01},
    ])
    optimizer_name = "AdamW"

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE * 2, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False)

total_steps = math.ceil(len(train_loader) / GRAD_ACCUM) * EPOCHS
warmup_steps = max(1, int(total_steps * WARMUP_RATIO))
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
autocast_dtype = torch.bfloat16 if USE_BF16 else torch.float32
print("optimizer:", optimizer_name, "steps:", total_steps, "warmup:", warmup_steps, "bf16:", USE_BF16)

model.safetensors:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


optimizer: StableAdamW steps: 5960 warmup: 298 bf16: True


In [7]:
try:
    # Check if 'model' exists and print its basic properties
    print(f"✅ Model successfully initialized!")
    print(f"Type: {type(model).__name__}")
    print(f"Device: {next(model.parameters()).device}")
except NameError:
    print("❌ Model is NOT initialized. Please make sure you successfully ran the cell above.")

✅ Model successfully initialized!
Type: MultiTaskModernBERT
Device: cuda:0


In [8]:
# 7. Evaluation helpers with dev-tuned hierarchy-aware inference
from sklearn.metrics import accuracy_score, classification_report, f1_score
from tqdm.auto import tqdm

def hierarchy_scores(outputs, lambda_group=0.0, lambda_sector=0.0):
    industry_logp = F.log_softmax(outputs["industry_logits"], dim=-1)
    group_logp = F.log_softmax(outputs["group_logits"], dim=-1)
    sector_logp = F.log_softmax(outputs["sector_logits"], dim=-1)
    return (
        industry_logp
        + lambda_group * group_logp[:, industry_to_group_idx]
        + lambda_sector * sector_logp[:, industry_to_sector_idx]
    )

def collect_predictions(loader, lambda_group=0.0, lambda_sector=0.0, return_probs=False):
    model.eval()
    sector_true, sector_pred = [], []
    group_true, group_pred = [], []
    industry_true, industry_pred = [], []
    industry_conf = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="eval", leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            sector_idx = batch["sector_idx"].to(device)
            group_idx = batch["group_idx"].to(device)
            industry_idx = batch["industry_idx"].to(device)
            with torch.autocast(device_type="cuda", dtype=autocast_dtype, enabled=USE_BF16):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                scores = hierarchy_scores(outputs, lambda_group=lambda_group, lambda_sector=lambda_sector)
            probs = scores.softmax(dim=-1)
            pred = scores.argmax(dim=-1)
            sector_true.extend(sector_idx.cpu().tolist())
            sector_pred.extend(outputs["sector_logits"].argmax(dim=-1).cpu().tolist())
            group_true.extend(group_idx.cpu().tolist())
            group_pred.extend(outputs["group_logits"].argmax(dim=-1).cpu().tolist())
            industry_true.extend(industry_idx.cpu().tolist())
            industry_pred.extend(pred.cpu().tolist())
            industry_conf.extend(probs.max(dim=-1).values.cpu().tolist())
    true_codes = le_industry.inverse_transform(industry_true)
    pred_codes = le_industry.inverse_transform(industry_pred)
    result = {
        "sector_acc": accuracy_score(sector_true, sector_pred),
        "group_acc": accuracy_score(group_true, group_pred),
        "industry_acc": accuracy_score(industry_true, industry_pred),
        "industry_macro_f1": f1_score(true_codes, pred_codes, average="macro", zero_division=0),
        "true_codes": true_codes.tolist(),
        "pred_codes": pred_codes.tolist(),
        "confidence": industry_conf,
    }
    return result

def tune_hierarchy_on_dev():
    grid = []
    for lg in [0.0, 0.05, 0.10, 0.15, 0.20, 0.30]:
        for ls in [0.0, 0.03, 0.06, 0.10, 0.15]:
            metrics = collect_predictions(dev_loader, lambda_group=lg, lambda_sector=ls)
            grid.append({"lambda_group": lg, "lambda_sector": ls, "macro_f1": metrics["industry_macro_f1"]})
            print(f"dev hierarchy lg={lg:.2f} ls={ls:.2f} f1={metrics['industry_macro_f1']*100:.2f}%")
    best = max(grid, key=lambda row: row["macro_f1"])
    print("best hierarchy:", best)
    return best, grid



In [9]:
# 8. Training loop
best_macro_f1 = -1.0
history = []
epochs_without_improvement = 0

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    started = time.time()
    progress = tqdm(train_loader, desc=f"train epoch {epoch + 1}/{EPOCHS}")

    for step, batch in enumerate(progress, start=1):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        sector_idx = batch["sector_idx"].to(device)
        group_idx = batch["group_idx"].to(device)
        industry_idx = batch["industry_idx"].to(device)

        with torch.autocast(device_type="cuda", dtype=autocast_dtype, enabled=USE_BF16):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = (
                alpha * loss_sector(outputs["sector_logits"], sector_idx)
                + beta_loss * loss_group(outputs["group_logits"], group_idx)
                + gamma * loss_industry(outputs["industry_logits"], industry_idx)
            )
            loss = loss / GRAD_ACCUM

        loss.backward()
        epoch_loss += loss.item() * GRAD_ACCUM

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        progress.set_postfix({"loss": round(epoch_loss / step, 4)})

    metrics = collect_predictions(dev_loader, lambda_group=0.0, lambda_sector=0.0)
    record = {
        "epoch": epoch + 1,
        "train_loss": epoch_loss / max(1, len(train_loader)),
        "dev_sector_acc": metrics["sector_acc"],
        "dev_group_acc": metrics["group_acc"],
        "dev_industry_acc": metrics["industry_acc"],
        "dev_industry_macro_f1": metrics["industry_macro_f1"],
        "elapsed_s": time.time() - started,
    }
    history.append(record)
    print(json.dumps(record, indent=2))

    checkpoint_dir = OUT_DIR / f"epoch_{epoch + 1}"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), checkpoint_dir / "model_state.pt")
    tokenizer.save_pretrained(checkpoint_dir)
    with open(checkpoint_dir / "metrics.json", "w", encoding="utf-8") as handle:
        json.dump(record, handle, indent=2)

    if metrics["industry_macro_f1"] > best_macro_f1 + MIN_DEV_F1_DELTA:
        best_macro_f1 = metrics["industry_macro_f1"]
        epochs_without_improvement = 0
        torch.save(model.state_dict(), OUT_DIR / "best_model_state.pt")
        with open(OUT_DIR / "best_metrics.json", "w", encoding="utf-8") as handle:
            json.dump(record, handle, indent=2)
        print("New best checkpoint saved.")
    else:
        epochs_without_improvement += 1
        print(f"No meaningful dev F1 improvement for {epochs_without_improvement} epoch(s).")
        if not SMOKE_TEST and epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("Early stopping triggered.")
            break

with open(OUT_DIR / "train_history.json", "w", encoding="utf-8") as handle:
    json.dump(history, handle, indent=2)

train epoch 1/10:   0%|          | 0/4761 [00:00<?, ?it/s]

eval:   0%|          | 0/265 [00:00<?, ?it/s]

{
  "epoch": 1,
  "train_loss": 2.193111663999189,
  "dev_sector_acc": 0.8412098298676749,
  "dev_group_acc": 0.7530718336483931,
  "dev_industry_acc": 0.6377599243856332,
  "dev_industry_macro_f1": 0.59901461300623,
  "elapsed_s": 676.2075152397156
}
New best checkpoint saved.


train epoch 2/10:   0%|          | 0/4761 [00:00<?, ?it/s]

eval:   0%|          | 0/265 [00:00<?, ?it/s]

...

eval:   0%|          | 0/265 [00:00<?, ?it/s]

{
  "epoch": 8,
  "train_loss": 0.3128787185828542,
  "dev_sector_acc": 0.8716918714555766,
  "dev_group_acc": 0.7963137996219282,
  "dev_industry_acc": 0.7169187145557656,
  "dev_industry_macro_f1": 0.7049398138006636,
  "elapsed_s": 674.5629811286926
}
No meaningful dev F1 improvement for 2 epoch(s).


train epoch 9/10:   0%|          | 0/4761 [00:00<?, ?it/s]

eval:   0%|          | 0/265 [00:00<?, ?it/s]

{
  "epoch": 9,
  "train_loss": 0.2951228153746911,
  "dev_sector_acc": 0.8752362948960303,
  "dev_group_acc": 0.7963137996219282,
  "dev_industry_acc": 0.7138468809073724,
  "dev_industry_macro_f1": 0.7002670920171945,
  "elapsed_s": 675.0907657146454
}
No meaningful dev F1 improvement for 3 epoch(s).
Early stopping triggered.


In [10]:
# 9. Final dev tuning, locked test evaluation, and export
from google.colab import files

model.load_state_dict(torch.load(OUT_DIR / "best_model_state.pt", map_location=device))

if SMOKE_TEST:
    summary = {
        "model_name": MODEL_NAME,
        "run_name": RUN_NAME,
        "smoke_test": True,
        "best_dev_macro_f1": best_macro_f1,
        "notes": "Smoke test only. Official test intentionally not evaluated.",
    }
    with open(OUT_DIR / "smoke_summary.json", "w", encoding="utf-8") as handle:
        json.dump(summary, handle, indent=2)

    # Generate a simple markdown summary for smoke test
    with open(OUT_DIR / "summary_report.md", "w", encoding="utf-8") as f:
        f.write(f"# Smoke Test Summary\n\n* **Run Name:** {RUN_NAME}\n* **Model:** {MODEL_NAME}\n* **Best Dev Macro F1:** {best_macro_f1:.4f}\n")

else:
    hierarchy_best, hierarchy_grid = tune_hierarchy_on_dev()
    final_metrics = collect_predictions(
        test_loader,
        lambda_group=hierarchy_best["lambda_group"],
        lambda_sector=hierarchy_best["lambda_sector"],
    )
    true_codes = final_metrics["true_codes"]
    pred_codes = final_metrics["pred_codes"]
    counts = Counter(true_codes)
    top10 = [code for code, _ in counts.most_common(10)]
    top10_f1 = f1_score(true_codes, pred_codes, average=None, labels=top10, zero_division=0)
    top10_pass = int(sum(score > 0.85 for score in top10_f1))
    tail = [code for code, support in counts.items() if support <= 50]
    tail_f1 = f1_score(true_codes, pred_codes, average="macro", labels=tail, zero_division=0) if tail else 0.0

    summary = {
        "version": "modernbert-v3-native-fulltext",
        "model_name": MODEL_NAME,
        "run_name": RUN_NAME,
        "smoke_test": False,
        "input_contract": "Uses only task1 CSV text and mstar_code labels. No test-label restoration.",
        "leakage_report": leakage_report,
        "optimizer": optimizer_name,
        "epochs_requested": FULL_EPOCHS,
        "epochs_ran": len(history),
        "batch_size": BATCH_SIZE,
        "gradient_accumulation": GRAD_ACCUM,
        "max_len": MAX_LEN,
        "best_dev_macro_f1": float(best_macro_f1),
        "dev_tuned_hierarchy": hierarchy_best,
        "sector_acc": final_metrics["sector_acc"],
        "group_acc": final_metrics["group_acc"],
        "industry_acc": final_metrics["industry_acc"],
        "industry_macro_f1": final_metrics["industry_macro_f1"],
        "tail_f1": float(tail_f1),
        "tail_class_count": len(tail),
        "top10_pass": top10_pass,
        "top10_breakdown": [
            {"code": code, "f1": float(score), "support": int(counts[code])}
            for code, score in zip(top10, top10_f1)
        ],
        "loss_weights": {"sector": alpha, "group": beta_loss, "industry": gamma},
    }

    with open(OUT_DIR / "final_summary.json", "w", encoding="utf-8") as handle:
        json.dump(summary, handle, indent=2)
    with open(OUT_DIR / "hierarchy_grid.json", "w", encoding="utf-8") as handle:
        json.dump(hierarchy_grid, handle, indent=2)
    with open(OUT_DIR / "leakage_audit.json", "w", encoding="utf-8") as handle:
        json.dump(leakage_report, handle, indent=2)

    np.save(OUT_DIR / "industry_classes.npy", le_industry.classes_)
    np.save(OUT_DIR / "group_classes.npy", le_group.classes_)
    np.save(OUT_DIR / "sector_classes.npy", le_sector.classes_)

    pd.DataFrame({
        "true_code": true_codes,
        "pred_code": pred_codes,
        "confidence": final_metrics["confidence"],
    }).to_csv(OUT_DIR / "test_predictions.csv", index=False)

    report = classification_report(true_codes, pred_codes, output_dict=True, zero_division=0)
    with open(OUT_DIR / "classification_report.json", "w", encoding="utf-8") as handle:
        json.dump(report, handle, indent=2)

    # Generate a human-readable markdown summary document
    summary_doc_content = f"""# Training Summary Report: {RUN_NAME}

## Model & Architecture
* **Model Name:** {MODEL_NAME}
* **Max Sequence Length:** {MAX_LEN}
* **Optimizer:** {optimizer_name}
* **Epochs (Requested / Ran):** {FULL_EPOCHS} / {len(history)}
* **Batch Size:** {BATCH_SIZE} (with Grad Accumulation: {GRAD_ACCUM})

## Performance Metrics
* **Best Dev Macro F1:** {best_macro_f1:.4f}
* **Test Sector Accuracy:** {final_metrics['sector_acc']:.4f}
* **Test Group Accuracy:** {final_metrics['group_acc']:.4f}
* **Test Industry Accuracy:** {final_metrics['industry_acc']:.4f}
* **Test Industry Macro F1:** {final_metrics['industry_macro_f1']:.4f}

## Hierarchy Tuning
* **Best Lambda Group:** {hierarchy_best['lambda_group']}
* **Best Lambda Sector:** {hierarchy_best['lambda_sector']}

## Tail & Top Class Performance
* **Tail F1 (<=50 samples):** {tail_f1:.4f} across {len(tail)} classes
* **Top 10 Classes Pass Rate (>0.85 F1):** {top10_pass}/10

*Input Contract:* {summary['input_contract']}
"""
    with open(OUT_DIR / "summary_report.md", "w", encoding="utf-8") as f:
        f.write(summary_doc_content)

    print(json.dumps(summary, indent=2))
    print(f"\nSummary markdown document saved to: {OUT_DIR / 'summary_report.md'}")

zip_path = Path("/content") / f"{RUN_NAME}_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in OUT_DIR.rglob("*"):
        zf.write(path, path.relative_to(OUT_DIR.parent))

print("Downloading:", zip_path)
files.download(str(zip_path))


eval:   0%|          | 0/265 [00:00<?, ?it/s]

dev hierarchy lg=0.00 ls=0.00 f1=70.66%


eval:   0%|          | 0/265 [00:00<?, ?it/s]

dev hierarchy lg=0.00 ls=0.03 f1=70.58%


eval:   0%|          | 0/265 [00:00<?, ?it/s]

...

dev hierarchy lg=0.30 ls=0.15 f1=70.65%
best hierarchy: {'lambda_group': 0.3, 'lambda_sector': 0.03, 'macro_f1': 0.7080507740933875}


eval:   0%|          | 0/670 [00:00<?, ?it/s]

{
  "version": "modernbert-v3-native-fulltext",
  "model_name": "answerdotai/ModernBERT-large",
  "run_name": "modernbert_gecs_v3b_fulltext_stable",
  "smoke_test": false,
  "input_contract": "Uses only task1 CSV text and mstar_code labels. No test-label restoration.",
  "leakage_report": {
    "train_rows": 42319,
    "test_rows": 10717,
    "exact_text_overlap_count": 248,
    "exact_text_overlap_rate_vs_test": 0.02314080432956984,
    "note": "V3 does not use test labels during training or dev tuning. Exact overlap should be investigated if non-zero."
  },
  "optimizer": "StableAdamW",
  "epochs_requested": 10,
  "epochs_ran": 9,
  "batch_size": 8,
  "gradient_accumulation": 8,
  "max_len": 512,
  "best_dev_macro_f1": 0.7065596085865096,
  "dev_tuned_hierarchy": {
    "lambda_group": 0.3,
    "lambda_sector": 0.03,
    "macro_f1": 0.7080507740933875
  },
  "sector_acc": 0.8680600914435009,
  "group_acc": 0.7939721937109265,
  "industry_acc": 0.7225902771297937,
  "industry_macro_f1"

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Backup to Google Drive (Recommended for large files)
Run this cell to copy the model outputs to your Google Drive instead of downloading them through the browser.

In [15]:
# Step 1: Rescue the small file (Download directly)
from google.colab import files
print("Downloading test_predictions.csv...")
files.download("/content/modernbert_gecs_v3b_fulltext_stable/test_predictions.csv")

# Step 2: Rescue the minimal model weights to Google Drive
from google.colab import drive
import shutil, os, glob

drive.mount('/content/drive')

src = "/content/modernbert_gecs_v3b_fulltext_stable"
dst = "/content/drive/MyDrive/v3_minimal"
os.makedirs(dst, exist_ok=True)

print(f"\nCopying essential files to {dst}...")
shutil.copy(f"{src}/best_model_state.pt", f"{dst}/best_model_state.pt")
shutil.copy(f"{src}/final_summary.json",   f"{dst}/final_summary.json")

# Tokenizer — pick the latest epoch dir
ep_dirs = sorted(glob.glob(f"{src}/epoch_*"))
if ep_dirs:
    ep = ep_dirs[-1]
    shutil.copy(f"{ep}/tokenizer.json",        f"{dst}/tokenizer.json")
    shutil.copy(f"{ep}/tokenizer_config.json", f"{dst}/tokenizer_config.json")

print("Saved essential files to Drive:", os.listdir(dst))

# Step 3: Free the Colab disk
# IMPORTANT: Uncomment the next two lines ONLY AFTER you verify the files are safely in your Drive!
# shutil.rmtree("/content/modernbert_gecs_v3b_fulltext_stable")
# print("\nCleared local storage.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Copying essential files to /content/drive/MyDrive/v3_minimal...
Saved essential files to Drive: ['best_model_state.pt', 'final_summary.json', 'tokenizer.json', 'tokenizer_config.json']
